In [38]:
import nibabel as nib
import numpy as np
from skimage.transform import resize

def load_nifti(path):
    return nib.load(path).get_fdata()

def preprocess(volume):
    volume = resize(volume, (16,16,16))   # 🔥 better than 8x8x8
    volume = (volume - np.min(volume)) / (np.max(volume) - np.min(volume))
    return volume

In [39]:
def extract_features(volume):
    return np.array([
        np.mean(volume),
        np.std(volume),
        np.max(volume),
        np.min(volume),
        np.percentile(volume, 25),
        np.percentile(volume, 50),
        np.percentile(volume, 75),
        np.var(volume),
        np.sum(volume),
        np.median(volume)
    ])

In [40]:
import numpy as np

def augment(volume):
    vol = volume.copy()

    # small noise
    noise = np.random.normal(0, 0.01, vol.shape)
    vol = vol + noise

    # random flip
    if np.random.rand() > 0.5:
        vol = np.flip(vol, axis=0)

    # slight scaling
    scale = np.random.uniform(0.9, 1.1)
    vol = vol * scale

    return vol

In [41]:
def load_dataset(paths):
    X, y = [], []

    label_map = {
        "healthy": 0,
        "alzheimer_early": 1,
        "alzheimer_diagnosed": 2,
        "parkinson_early": 3,
        "parkinson_diagnosed": 4
    }

    for label_name, folder in paths.items():
        for f in os.listdir(folder):
            if f.endswith(".nii") or f.endswith(".nii.gz"):

                vol = preprocess(load_nifti(os.path.join(folder, f)))

                # original
                feat = extract_features(vol)
                X.append(feat)
                y.append(label_map[label_name])

                # augmented
                aug_vol = augment(vol)
                aug_feat = extract_features(aug_vol)
                X.append(aug_feat)
                y.append(label_map[label_name])

    return np.array(X), np.array(y)   # 🔥 MUST BE PRESENT

In [42]:
features, labels = load_dataset(paths)
print(len(labels))

180


In [43]:
paths = {
    "healthy": "alzimers/cn",
    "alzheimer_early": "alzimers/mci",
    "alzheimer_diagnosed": "alzimers/ad",
    "parkinson_early": "parkinsons/detect",
    "parkinson_diagnosed": "parkinsons/disease"
}

features, labels = load_dataset(paths)

In [44]:
from sklearn.preprocessing import StandardScaler
import torch

scaler = StandardScaler()
features = scaler.fit_transform(features)

features = torch.tensor(features, dtype=torch.float)
labels = torch.tensor(labels, dtype=torch.long)

In [45]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    list(range(len(labels))),
    test_size=0.2,
    stratify=labels,
    random_state=42
)

train_mask = torch.zeros(len(labels), dtype=torch.bool)
test_mask = torch.zeros(len(labels), dtype=torch.bool)

train_mask[train_idx] = True
test_mask[test_idx] = True

In [46]:
from sklearn.neighbors import kneighbors_graph

A = kneighbors_graph(features.numpy(), 5, mode='distance')

edge_index = []
edge_weight = []

rows, cols = A.nonzero()

for i, j in zip(rows, cols):
    edge_index.append([i, j])
    edge_weight.append(A[i, j])

edge_index = torch.tensor(edge_index).t().contiguous()
edge_weight = torch.tensor(edge_weight, dtype=torch.float)

from torch_geometric.data import Data

graph_data = Data(
    x=features,
    edge_index=edge_index,
    edge_weight=edge_weight,
    y=labels
)

In [47]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(10, 32)
        self.conv2 = GCNConv(32, 16)
        self.fc = torch.nn.Linear(16, 5)

    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index, data.edge_weight))
        x = F.relu(self.conv2(x, data.edge_index, data.edge_weight))
        x = self.fc(x)
        return F.log_softmax(x, dim=1)

In [56]:
model = GNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(30):
    model.train()
    optimizer.zero_grad()

    out = model(graph_data)
    loss = F.nll_loss(out[train_mask], labels[train_mask])

    loss.backward()
    optimizer.step()

    print(epoch, loss.item())

0 1.5957789421081543
1 1.5336352586746216
2 1.4796371459960938
3 1.4343799352645874
4 1.3987410068511963
5 1.3716164827346802
6 1.3474253416061401
7 1.3235642910003662
8 1.2996761798858643
9 1.2765183448791504
10 1.2553752660751343
11 1.2372666597366333
12 1.2223000526428223
13 1.2097322940826416
14 1.199349284172058
15 1.189695954322815
16 1.179114818572998
17 1.1680502891540527
18 1.1574405431747437
19 1.1482665538787842
20 1.1398544311523438
21 1.1322017908096313
22 1.1240544319152832
23 1.1155238151550293
24 1.1079292297363281
25 1.1001240015029907
26 1.092146396636963
27 1.084519624710083
28 1.07765531539917
29 1.0712437629699707


In [57]:
model.eval()
out = model(graph_data)

pred = out[test_mask].argmax(dim=1)
acc = (pred == labels[test_mask]).sum().item() / test_mask.sum().item()

print("Test Accuracy:", acc)

Test Accuracy: 0.3333333333333333


In [50]:
from collections import Counter
print(Counter(labels.tolist()))

Counter({0: 60, 1: 30, 2: 30, 3: 30, 4: 30})


In [51]:
def predict_new(file_path):
    vol = preprocess(load_nifti(file_path))
    feat = extract_features(vol)

    feat = scaler.transform([feat])
    feat = torch.tensor(feat, dtype=torch.float)

    model.eval()

    # 🔥 pass as single node (no graph needed)
    x = feat
    x = F.relu(model.conv1(x, edge_index, edge_weight))
    x = F.relu(model.conv2(x, edge_index, edge_weight))
    x = model.fc(x)

    pred = x.argmax(dim=1).item()

    classes = [
        "Healthy",
        "Alzheimer Early",
        "Alzheimer Diagnosed",
        "Parkinson Early",
        "Parkinson Diagnosed"
    ]

    print("Prediction:", classes[pred])